In [7]:
# Importações estritamente necessárias

from notebookutils import mssparkutils
from pyspark.sql.utils import AnalysisException


# VALIDAÇÃO DE ARQUIVOS E LEITURA (Camada Bronze)

origem_relativa = "Files/Cotacoes/Novos"
destino_relativo = "Files/Cotacoes/Carregados"

# Bloco try-except para tratar de forma elegante caso a pasta esteja vazia ou não exista

try:
    df_raw = spark.read.parquet(f"{origem_relativa}/*.parquet")
except AnalysisException:
    print("Nenhum arquivo novo encontrado para processamento. Encerrando execução com sucesso.")
    mssparkutils.notebook.exit("Sem arquivos novos") # Encerra o notebook sem falhar o pipeline

# Cria a view temporária se houverem dados

df_raw.createOrReplaceTempView("vw_raw_cotacoes")

# Padroniza colunas, tipagem e remove duplicatas

df_clean = spark.sql("""
    SELECT
        cotacaoCompra AS Cotacao,
        CAST(dataHoraCotacao AS DATE) AS Data,
        moeda AS Moeda
    FROM
        vw_raw_cotacoes
""").dropDuplicates(["Moeda", "Data"])

df_clean.createOrReplaceTempView("vw_novas_cotacoes")

# Garante a existência da tabela Delta particionada

spark.sql("""
    CREATE TABLE IF NOT EXISTS cotacoes (
        Cotacao DOUBLE,
        Data DATE,
        Moeda STRING
    )
    USING DELTA
    PARTITIONED BY (Moeda)
""")

# CARGA INCREMENTAL (Merge / Upsert)

spark.sql("""
    MERGE INTO cotacoes AS e 
    USING vw_novas_cotacoes AS n
    ON e.Moeda = n.Moeda AND e.Data = n.Data
    
    WHEN NOT MATCHED THEN
        INSERT (Cotacao, Data, Moeda)
        VALUES (n.Cotacao, n.Data, n.Moeda)
""")

# Garante que a pasta de destino exista usando o caminho relativo

if not mssparkutils.fs.exists(destino_relativo):
    mssparkutils.fs.mkdirs(destino_relativo)

# Lista os arquivos e move iterativamente

arquivos = mssparkutils.fs.ls(origem_relativa)

for arquivo in arquivos:
    # Movimentação usando apenas os caminhos relativos

    caminho_origem = f"{origem_relativa}/{arquivo.name}"
    caminho_destino = f"{destino_relativo}/{arquivo.name}"
    
    mssparkutils.fs.mv(caminho_origem, caminho_destino)

StatementMeta(, f197a4d1-0387-49b4-83ea-225ca23f969a, 9, Finished, Available, Finished, False)

Nenhum arquivo novo encontrado para processamento. Encerrando execução com sucesso.
ExitValue: Sem arquivos novos